In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys

!{sys.executable} -m pip install rasterio numpy pyarrow

In [ ]:
import rasterio
from rasterio.windows import Window
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import glob
import os

def process_polygon_to_parquet(tiff_path, parquet_output_path, chunk_size=2048):
    # 1. Setup Schema for Parquet (No 'target' column)
    # Matching the exact bands exported from the GEE script
    target_bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']

    fields = [pa.field(b, pa.float32()) for b in target_bands]
    fields.extend([
        pa.field('longitude', pa.float32()),
        pa.field('latitude', pa.float32()),
        pa.field('poly_name', pa.string()), # Extracted from filename
        pa.field('year', pa.int32()),       # Extracted from filename
        pa.field('month', pa.int32())       # Extracted from filename
    ])
    schema = pa.schema(fields)

    # 2. Extract Metadata from Filename (Expected: SOC_Poly_1_2025_01.tif)
    basename = os.path.basename(tiff_path).replace('.tif', '')
    parts = basename.split('_')
    try:
        poly_name = parts[2]
        year = int(parts[3])
        month = int(parts[4])
    except IndexError:
        poly_name = "Unknown"
        year = 0
        month = 0

    # 3. Read Data and Write to Parquet
    with rasterio.open(tiff_path) as src:
        # GEE exports bands in the exact order we defined
        band_indices = {band: idx + 1 for idx, band in enumerate(target_bands)}

        with pq.ParquetWriter(parquet_output_path, schema, compression='snappy') as writer:
            for row_off in range(0, src.height, chunk_size):
                for col_off in range(0, src.width, chunk_size):

                    window = Window(col_off, row_off,
                                    min(chunk_size, src.width - col_off),
                                    min(chunk_size, src.height - row_off))

                    valid_mask = np.zeros(window.height * window.width, dtype=bool)
                    data_dict = {}

                    # Read all bands
                    for band_name, idx in band_indices.items():
                        band_data = src.read(idx, window=window).flatten()
                        # Valid pixels are those greater than 0 (not masked by GEE)
                        valid_mask |= (band_data > 0)
                        data_dict[band_name] = band_data

                    # If the entire chunk is empty (e.g., heavily cloud-masked), skip it
                    if not valid_mask.any():
                        continue

                    # Filter out invalid/empty pixels
                    for b in target_bands:
                        data_dict[b] = data_dict[b][valid_mask].astype(np.float32)

                    # Calculate precise lat/lon coordinates
                    rows, cols = np.meshgrid(
                        np.arange(row_off, row_off + window.height),
                        np.arange(col_off, col_off + window.width),
                        indexing='ij'
                    )
                    valid_rows = rows.flatten()[valid_mask]
                    valid_cols = cols.flatten()[valid_mask]
                    xs, ys = rasterio.transform.xy(src.transform, valid_rows, valid_cols)

                    data_dict['longitude'] = np.array(xs, dtype=np.float32)
                    data_dict['latitude'] = np.array(ys, dtype=np.float32)

                    # Add Tracking Metadata
                    num_valid = valid_mask.sum()
                    data_dict['poly_name'] = np.array([poly_name] * num_valid)
                    data_dict['year'] = np.array([year] * num_valid, dtype=np.int32)
                    data_dict['month'] = np.array([month] * num_valid, dtype=np.int32)

                    # Ensure column order matches schema exactly
                    col_order = target_bands + ['longitude', 'latitude', 'poly_name', 'year', 'month']
                    ordered_dict = {col: data_dict[col] for col in col_order}

                    table = pa.Table.from_pydict(ordered_dict, schema=schema)
                    writer.write_table(table)


def batch_convert_gee_exports():
    """Automates the conversion of all polygon TIFFs to Parquet."""

    # Setup your Drive paths
    input_dir = "/content/drive/MyDrive/bare_soil/Monthly_Median_Exports"
    out_dir = "/content/drive/MyDrive/bare_soil/parquet_exports"

    os.makedirs(out_dir, exist_ok=True)

    # Grab all TIFFs generated by the GEE script
    tiff_files = glob.glob(os.path.join(input_dir, "SOC_Poly_*.tif"))

    if not tiff_files:
        print(f"❌ No TIFF files found in: {input_dir}")
        return

    print(f"🔍 Found {len(tiff_files)} TIFF files to process.")
    print(f"==================================================")

    for tiff_path in tiff_files:
        filename = os.path.basename(tiff_path)
        parquet_name = filename.replace('.tif', '.parquet')
        parquet_path = os.path.join(out_dir, parquet_name)

        if os.path.exists(parquet_path):
            print(f"⏭️ SKIPPED (Already exists): {parquet_name}")
            continue

        try:
            process_polygon_to_parquet(tiff_path, parquet_path)

            # Since some files might be empty due to 100% cloud cover, handle 0 byte files cleanly
            if os.path.exists(parquet_path):
                size_kb = os.path.getsize(parquet_path) / 1024
                if size_kb < 1:  # Parquet metadata is usually at least a few KB
                    print(f"⚠️ {filename} processed, but resulted in an empty file (likely 100% cloudy).")
                else:
                    print(f"✅ SUCCESS: {filename} -> {size_kb:.1f} KB")

        except Exception as e:
            print(f"❌ FAILED to process {filename}: {e}")

if __name__ == "__main__":
    batch_convert_gee_exports()

In [ ]:
import pandas as pd
import glob
import os

def append_soc_targets():
    # 1. Define your Ground Truth Dictionary
    # Keys are strings to match the 'poly_name' column extracted from the filenames
    soc_mapping = {
        "1": 0.43, "2": 0.28, "3": 0.62, "4": 0.14, "5": 0.13,
        "6": 0.56, "7": 0.53, "8": 0.50, "9": 0.63, "10": 0.69,
        "11": 0.30, "12": 0.59, "13": 0.57, "14": 0.61, "15": 0.63,
        "16": 0.76, "17": 0.68, "18": 0.50, "19": 0.46, "20": 0.39,
        "21": 0.64, "22": 0.55, "23": 0.28, "24": 0.85, "25": 0.74,
        "26": 0.62, "27": 0.51, "28": 0.63, "29": 0.61, "30": 0.58,
        "31": 1.01, "32": 0.53, "33": 0.26, "34": 0.31, "35": 1.01,
        "36": 0.63, "37": 0.61, "38": 0.40, "39": 0.55, "40": 0.29
    }

    # 2. Setup your Directories
    input_dir = r"/content/drive/MyDrive/bare_soil/soc/parquet_exports"

    # It's safer to save to a new folder so you don't corrupt your base data
    out_dir = r"/content/drive/MyDrive/bare_soil/soc/parquet_with_targets"
    os.makedirs(out_dir, exist_ok=True)

    # 3. Grab all Parquet files
    parquet_files = glob.glob(os.path.join(input_dir, "*.parquet"))

    if not parquet_files:
        print(f"❌ No Parquet files found in: {input_dir}")
        return

    print(f"🔍 Found {len(parquet_files)} Parquet files. Appending SOC targets...")
    print("=" * 50)

    # 4. Process each file
    for file_path in parquet_files:
        filename = os.path.basename(file_path)
        out_path = os.path.join(out_dir, filename)

        try:
            # Read the parquet
            df = pd.read_parquet(file_path)

            # Map the SOC values based on the poly_name column
            # We use df['poly_name'].astype(str) just to guarantee data type safety
            df['target_soc'] = df['poly_name'].astype(str).map(soc_mapping)

            # Check if any rows failed to map (e.g., if a poly_name wasn't in the dict)
            missing_targets = df['target_soc'].isna().sum()
            if missing_targets > 0:
                print(f"⚠️ WARNING: {filename} has {missing_targets} rows with unmapped SOC values!")
9
            # Reorder columns slightly so target_soc is at the front (optional, but clean for ML)
            cols = df.columns.tolist()
            cols.insert(0, cols.pop(cols.index('target_soc')))
            df = df[cols]

            # Save the updated DataFrame
            df.to_parquet(out_path, engine='pyarrow', compression='snappy', index=False)

            print(f"✅ Processed: {filename}")

        except Exception as e:
            print(f"❌ FAILED to process {filename}: {e}")

    print("\n🎉 All files updated and saved to:", out_dir)

if __name__ == "__main__":
    append_soc_targets()

In [ ]:
import pandas as pd
import glob
import os
import gc
from sklearn.model_selection import train_test_split

def process_safely_and_save():
    # 1. Your SOC Ground Truth Dictionary
    soc_mapping = {
        "1": 0.43, "2": 0.28, "3": 0.62, "4": 0.14, "5": 0.13,
        "6": 0.56, "7": 0.53, "8": 0.50, "9": 0.63, "10": 0.69,
        "11": 0.30, "12": 0.59, "13": 0.57, "14": 0.61, "15": 0.63,
        "16": 0.76, "17": 0.68, "18": 0.50, "19": 0.46, "20": 0.39,
        "21": 0.64, "22": 0.55, "23": 0.28, "24": 0.85, "25": 0.74,
        "26": 0.62, "27": 0.51, "28": 0.63, "29": 0.61, "30": 0.58,
        "31": 1.01, "32": 0.53, "33": 0.26, "34": 0.31, "35": 1.01,
        "36": 0.63, "37": 0.61, "38": 0.40, "39": 0.55, "40": 0.29
    }

    # 2. Setup Directories
    bare_dir = "/content/drive/MyDrive/bare_soil/soc/parquet_exports_bare"
    out_dir = "/content/drive/MyDrive/bare_soil/soc/final_ml_datasets"
    os.makedirs(out_dir, exist_ok=True)

    files = glob.glob(os.path.join(bare_dir, "*.parquet"))
    if not files:
        raise FileNotFoundError(f"No parquet files found in {bare_dir}")

    # 3. PRE-COMPUTE THE SPLIT (Extremely Memory Efficient)
    unique_polys = list(soc_mapping.keys())
    # Hold out 20% of the polygons for testing
    train_polys, test_polys = train_test_split(unique_polys, test_size=0.2, random_state=42)

    print(f"Training on {len(train_polys)} polygons, Testing on {len(test_polys)} polygons.")

    # Lists to hold the chunks
    train_base_list, test_base_list = [], []
    train_bare_list, test_bare_list = [], []

    print(f"Processing {len(files)} files iteratively to prevent OOM...")

    # 4. ITERATE AND CHUNK
    for f in files:
        df = pd.read_parquet(f)

        # Map target and drop unmapped
        df['target_soc'] = df['poly_name'].astype(str).map(soc_mapping)
        df.dropna(subset=['target_soc'], inplace=True)

        if df.empty:
            continue

        # Split into Train and Test based on our pre-computed lists
        is_train = df['poly_name'].astype(str).isin(train_polys)

        train_chunk = df[is_train]
        test_chunk = df[~is_train]

        # Append to Base lists and Bare lists simultaneously
        if not train_chunk.empty:
            train_base_list.append(train_chunk)
            train_bare_list.append(train_chunk[train_chunk['prediction'] == 1])

        if not test_chunk.empty:
            test_base_list.append(test_chunk)
            test_bare_list.append(test_chunk[test_chunk['prediction'] == 1])

        # Delete the temp variables and force garbage collection to keep RAM clear
        del df, train_chunk, test_chunk, is_train
        gc.collect()

    # 5. CONCATENATE AND SAVE ONE-BY-ONE
    # Doing this sequentially guarantees we never hold all 4 massive dataframes in RAM at once.
    print("\nSaving final datasets to Google Drive...")

    # Dataset 1: Base Train
    print("Saving Train Base...")
    train_base = pd.concat(train_base_list, ignore_index=True)
    train_base.to_parquet(os.path.join(out_dir, "train_base.parquet"), index=False)
    del train_base, train_base_list
    gc.collect()

    # Dataset 2: Base Test
    print("Saving Test Base...")
    test_base = pd.concat(test_base_list, ignore_index=True)
    test_base.to_parquet(os.path.join(out_dir, "test_base.parquet"), index=False)
    del test_base, test_base_list
    gc.collect()

    # Dataset 3: Bare Train
    print("Saving Train Bare...")
    train_bare = pd.concat(train_bare_list, ignore_index=True)
    train_bare.to_parquet(os.path.join(out_dir, "train_bare.parquet"), index=False)
    del train_bare, train_bare_list
    gc.collect()

    # Dataset 4: Bare Test
    print("Saving Test Bare...")
    test_bare = pd.concat(test_bare_list, ignore_index=True)
    test_bare.to_parquet(os.path.join(out_dir, "test_bare.parquet"), index=False)
    del test_bare, test_bare_list
    gc.collect()

    print(f"✅ All datasets safely built and saved to: {out_dir}")

# Execute the pipeline
if __name__ == "__main__":
    process_safely_and_save()

Training on 32 polygons, Testing on 8 polygons.
Processing 478 files iteratively to prevent OOM...

Saving final datasets to Google Drive...
Saving Train Base...
Saving Test Base...
Saving Train Bare...
Saving Test Bare...
✅ All datasets safely built and saved to: /content/drive/MyDrive/bare_soil/soc/final_ml_datasets
